# Part V: LLM Critique

The LLM was asked to generate Python code for the half-circle to Gaussian optimal transport experiment.

I then inspected the code and numerical results according to the five criteria in the assignment.

# 1. Does the LLM Normalize the Densities Correctly?

The source density is

$$
\rho_0(x)
=
\frac{2}{\pi}\sqrt{1-x^2},
\qquad
-1\le x\le1.
$$

This formula is already analytically normalized, since

$$
\int_{-1}^{1}
\rho_0(x)\,dx
=
1.
$$

The code did not unnecessarily rescale the analytic density. Instead, it checked its numerical integral using trapezoidal quadrature.

The computed value was

$$
\int_{-1}^{1}\rho_0(x)\,dx
\approx
0.999995815.
$$

Therefore, the numerical error was approximately

$$
4.19\times10^{-6}.
$$

This is very small and is consistent with finite-grid quadrature error near the square-root behavior at the endpoints.

For the target, the code used the standard Gaussian density

$$
\rho_1(y)
=
\frac{1}{\sqrt{2\pi}}
e^{-y^2/2}.
$$

The Gaussian is exactly normalized on

$$
\mathbb R.
$$

Importantly, the code did not incorrectly renormalize the Gaussian on the plotting interval

$$
[-6,6]
$$

and pretend that this finite interval was the entire real line.

Instead, it computed the amount of Gaussian mass captured on that interval:

$$
0.9999999980268233,
$$

and reported the missing tail mass as

$$
1.97318\times10^{-9}.
$$

Therefore, the density normalization was handled correctly.

The small discrepancy in the numerical source integral is a quadrature error rather than a mathematical normalization error.

# 2. Does the LLM Compute the CDFs Monotonically?

Yes.

For the source density, the numerical CDF was constructed using cumulative trapezoidal integration:

$$
F_0(x)
=
\int_{-1}^{x}\rho_0(s)\,ds.
$$

The code then normalized the final numerical CDF value to $1$.

The result satisfied

$$
F_0(-1)=0
$$

and

$$
F_0(1)=1.
$$

The code explicitly checked that

$$
F_0(x_{k+1})\ge F_0(x_k)
$$

throughout the grid.

The monotonicity test returned

$$
\text{True}.
$$

The numerical source CDF was also compared with the analytic formula

$$
F_0(x)
=
\frac12
+
\frac{1}{\pi}
\left(
\arcsin x
+
x\sqrt{1-x^2}
\right).
$$

The maximum difference was only

$$
2.03\times10^{-6}.
$$

For the Gaussian target, the code used the standard Gaussian CDF

$$
F_1(y)
=
\Phi(y),
$$

implemented using `norm.cdf`.

Its monotonicity check also returned

$$
\text{True}.
$$

Therefore, both CDFs were computed in a mathematically appropriate and monotone way.

# 3. Does the LLM Use a Generalized Inverse or Interpolation for the Quantile Function?

Yes.

The optimal one-dimensional transport map is

$$
T(x)
=
F_1^{-1}(F_0(x)).
$$

For a general CDF, the notation

$$
F^{-1}
$$

should be interpreted as a generalized inverse or quantile function rather than assuming that the CDF has an ordinary symbolic inverse.

The code addressed this correctly in two ways.

First, for the actual Gaussian transport map it used

$$
T(x)
=
\Phi^{-1}(F_0(x)),
$$

where `norm.ppf` provides the Gaussian quantile function.

Second, the code independently defined a numerical generalized-inverse procedure using interpolation of the sampled CDF.

Repeated CDF values were removed before interpolation to make the numerical inverse stable.

The interpolated inverse was then compared with the Gaussian quantile function.

The maximum difference was only

$$
1.33\times10^{-7}.
$$

Therefore, the numerical generalized inverse agrees very closely with the exact Gaussian quantile function.

The code also checked the defining relation

$$
\Phi(T(x))
=
F_0(x).
$$

The maximum error was

$$
2.22\times10^{-16},
$$

which is essentially machine precision.

Thus the LLM handled the quantile function correctly rather than treating the CDF as though it automatically had an ordinary algebraic inverse.

# 4. Does the LLM Handle Compact Support Correctly?

Yes.

The source density is compactly supported on

$$
[-1,1].
$$

The code explicitly sets

$$
\rho_0(x)=0
$$

outside this interval.

A more subtle issue occurs at the endpoints.

Since

$$
F_0(-1)=0,
$$

the transport map satisfies

$$
T(-1)
=
\Phi^{-1}(0)
=
-\infty.
$$

Similarly,

$$
F_0(1)=1
$$

implies

$$
T(1)
=
\Phi^{-1}(1)
=
+\infty.
$$

The code did not evaluate the numerical particle map exactly at

$$
x=\pm1.
$$

Instead, it used points slightly inside the support.

This avoids numerical infinities while preserving the correct mathematical endpoint behavior.

The endpoint experiment confirmed the divergence.

For example,

$$
T(0.9)
\approx
2.0815,
$$

$$
T(0.99)
\approx
3.2392,
$$

$$
T(0.9999)
\approx
4.8556,
$$

and

$$
T(1-10^{-8})
\approx
7.1025.
$$

Thus,

$$
T(x)\to+\infty
$$

as

$$
x\to1^-.
$$

By symmetry,

$$
T(x)\to-\infty
$$

as

$$
x\to-1^+.
$$

The code also examined the interpolated particle map

$$
X_t(x)
=
(1-t)x+tT(x).
$$

For every

$$
t>0,
$$

particles beginning increasingly close to the endpoint were transported increasingly far away.

Therefore, the code correctly reflects the fact that a compactly supported distribution can be transported into a distribution with unbounded support.

One small point worth noting is that the final diagnostic line

`Compact-support endpoint divergence demonstrated: True`

is hard-coded.

However, the actual endpoint table independently demonstrates the divergence, so the mathematical conclusion does not depend on that hard-coded statement.

# 5. Does the LLM Handle Gaussian Tails Honestly?

Yes, and this is one of the strongest parts of the implementation.

The standard Gaussian has support

$$
\mathbb R,
$$

so no finite computational grid can contain the entire distribution.

The code explicitly states that the interval

$$
[-6,6]
$$

is used only for plotting and numerical interpolation.

It does not claim that this interval contains all Gaussian probability.

Instead, it computes the missing probability mass.

For

$$
L=6,
$$

the omitted Gaussian mass was

$$
1.97\times10^{-9}.
$$

The code also performed a separate truncation study for

$$
L=2,3,4,5,6.
$$

The results showed:

| $L$ | Mass inside $[-L,L]$ | Mass outside |
| ---: | ---: | ---: |
| $2$ | $0.954500$ | $4.55\times10^{-2}$ |
| $3$ | $0.997300$ | $2.70\times10^{-3}$ |
| $4$ | $0.999937$ | $6.33\times10^{-5}$ |
| $5$ | $0.999999$ | $5.73\times10^{-7}$ |
| $6$ | approximately $1$ | approximately $1.97\times10^{-9}$ |

The code also compared the transport map for the full Gaussian with maps obtained from truncated Gaussian distributions.

For the central probability region

$$
0.001\le F_0(x)\le0.999,
$$

the maximum transport-map error was approximately

$$
1.1068
$$

for

$$
L=2,
$$

$$
0.2625
$$

for

$$
L=3,
$$

$$
0.00923
$$

for

$$
L=4,
$$

and only

$$
8.47\times10^{-5}
$$

for

$$
L=5.
$$

Therefore, the code clearly distinguishes the true Gaussian on

$$
\mathbb R
$$

from a finite truncated approximation.

It does not pretend that infinity fits on a finite computational grid.

# Overall Critique

For these five criteria, the LLM-generated code performed well.

It correctly:

- represented the normalized half-circle source density;
- treated the Gaussian as a probability density on all of $\mathbb R$;
- constructed monotone CDFs;
- used a genuine quantile function and numerical generalized inverse;
- handled the compact source support and divergent endpoint transport correctly;
- explicitly accounted for Gaussian probability outside finite numerical intervals.

The strongest numerical checks were

$$
\max
|\Phi(T(x))-F_0(x)|
\approx
2.22\times10^{-16},
$$

and

$$
\max
|\rho_{t=1}(y)-\rho_1(y)|
\approx
5.55\times10^{-17}.
$$

The intermediate probability masses also remained very close to one, with errors only on the order of

$$
10^{-6}.
$$

Therefore, I found no major mathematical error in this particular LLM-generated implementation.

The main lesson is still that the code should not be trusted only because it produces `PASS` statements.

For example, one diagnostic statement about endpoint divergence was hard-coded rather than automatically proved by the program.

The conclusion is reliable only because the underlying numerical values, formulas, and plots were checked independently.

Overall, the LLM was useful and mathematically careful in this experiment, but numerical verification remained necessary.